# Structural sensitivity analysis

Compare the original AlphaFold model of each monoterpene synthase with
its second- and third-ranked models. Retain alternative pockets with at
least 75% of the residues in the original pocket.

Structural changes are assessed using amino-acid composition, all 46
pocket features, and the 12 features used by the XGBoost classifier.
Leave-one-protein-out validation trains on `FullDataset.csv` from the
repository root and compares predictions for each withheld protein
across its original and alternative structures.


## Load the structural models


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from xgboost import XGBClassifier


FULL_DATASET_LOCATIONS = [
    Path("../../FullDataset.csv"),
    Path("FullDataset.csv"),
]
FULL_DATASET_FILE = next(
    (path.resolve() for path in FULL_DATASET_LOCATIONS if path.is_file()),
    None,
)
if FULL_DATASET_FILE is None:
    raise FileNotFoundError(
        "FullDataset.csv was not found in the ATC repository root. "
        "From the sensitivity folder, its path should be ../../FullDataset.csv."
    )
full_dataset = pd.read_csv(FULL_DATASET_FILE)
print(f"Training dataset: {FULL_DATASET_FILE}")

DATASET_NAME = "Structural_Sensitivity_Dataset.csv"
DATASET_LOCATIONS = [
    Path(DATASET_NAME),
    Path("SupplementaryInformation") / "Structural_Sensitivity" / DATASET_NAME,
]
DATASET_FILE = next(
    (path.resolve() for path in DATASET_LOCATIONS if path.is_file()),
    None,
)
if DATASET_FILE is None:
    raise FileNotFoundError(
        f"Place {DATASET_NAME} beside this notebook in "
        "SupplementaryInformation/Structural_Sensitivity/."
    )

MINIMUM_RESIDUE_RETENTION = 0.75
RANDOM_SEED = 42
SELECTED_FEATURES = [
    "Alanine - A",
    "Methionine - M",
    "Tryptophan - W",
    "Isoleucine - I",
    "Cysteine - C",
    "Histidine - H",
    "Asparagine - N",
    "Positive charge",
    "Negative charge",
    "Polar",
    "Non-polar",
    "Amino Acid based volume Score",
]
AMINO_ACID_FEATURES = [
    "Glycine - G", "Alanine - A", "Leucine - L", "Methionine - M",
    "Phenylalanine - F", "Tryptophan - W", "Lysine - K", "Glutamine - Q",
    "Glutamic Acid - E", "Serine - S", "Proline - P", "Valine - V",
    "Isoleucine - I", "Cysteine - C", "Tyrosine - Y", "Histidine - H",
    "Arginine - R", "Asparagine - N", "Aspartic Acid - D", "Threonine - T",
]

models = pd.read_csv(DATASET_FILE)
models = models.loc[models["BaseProtein"].ne("O22340")].copy()
if "Protein" not in full_dataset.columns:
    raise ValueError("FullDataset.csv must include a Protein column.")
full_dataset["BaseProtein"] = (
    full_dataset["Protein"].astype(str).str.replace(r"_[LC]$", "", regex=True)
)
full_dataset = full_dataset.loc[
    full_dataset["BaseProtein"].ne("O22340")
].copy()
metadata = {"Protein", "BaseProtein", "Rank", "Cyclical"}
features = [column for column in models.columns if column not in metadata]
required = metadata | set(AMINO_ACID_FEATURES) | set(SELECTED_FEATURES)
missing = sorted(required - set(models.columns))
if missing:
    raise ValueError(f"The structural dataset is missing columns: {missing}")
if len(features) != 46:
    raise ValueError(f"Expected 46 pocket features; found {len(features)}.")
missing_training_columns = sorted(
    {"Protein", "BaseProtein", "Cyclical", *features} - set(full_dataset.columns)
)
if missing_training_columns:
    raise ValueError(
        f"FullDataset.csv is missing columns: {missing_training_columns}"
    )

models["Rank"] = pd.to_numeric(models["Rank"], errors="raise").astype(int)
models["Cyclical"] = pd.to_numeric(models["Cyclical"], errors="raise").astype(int)
models[features] = models[features].apply(pd.to_numeric, errors="raise")
full_dataset["Cyclical"] = pd.to_numeric(
    full_dataset["Cyclical"], errors="raise"
).astype(int)
full_dataset[features] = full_dataset[features].apply(
    pd.to_numeric, errors="raise"
)
if models[features].isna().any().any():
    raise ValueError("Pocket features contain missing values.")
if not models["Rank"].isin([1, 2, 3]).all():
    raise ValueError("Model ranks must be 1, 2, or 3.")
if not models["Cyclical"].isin([0, 1]).all():
    raise ValueError("Class labels must be 0 or 1.")
if not full_dataset["Cyclical"].isin([0, 1]).all():
    raise ValueError("FullDataset.csv class labels must be 0 or 1.")
if full_dataset[features].isna().any().any():
    raise ValueError("FullDataset.csv pocket features contain missing values.")
if full_dataset["BaseProtein"].duplicated().any():
    raise ValueError("FullDataset.csv contains duplicate protein identifiers.")
if models.duplicated(["BaseProtein", "Rank"]).any():
    raise ValueError("A protein has duplicate model ranks.")
if models.groupby("BaseProtein")["Cyclical"].nunique().gt(1).any():
    raise ValueError("Models from the same protein have different class labels.")

originals = models.loc[models["Rank"].eq(1)].copy()
if set(originals["BaseProtein"]) != set(models["BaseProtein"]):
    raise ValueError("Every protein must have an original rank-1 model.")
if originals["Number of residues"].le(0).any():
    raise ValueError("Original pockets must contain at least one residue.")
missing_training_proteins = sorted(
    set(originals["BaseProtein"]) - set(full_dataset["BaseProtein"])
)
if missing_training_proteins:
    raise ValueError(
        "Original structural models are absent from FullDataset.csv: "
        f"{missing_training_proteins}"
    )

print(f"Structural models: {len(models)}")
print(f"Proteins: {len(originals)}")
print(f"Original models: {len(originals)}")
print(f"Full training dataset: {len(full_dataset)} proteins")


Training dataset: C:\Users\catha\OneDrive - The University of Manchester\Work\Reports\FullDataset.csv
Structural models: 257
Proteins: 86
Original models: 86
Full training dataset: 87 proteins


## Filter alternative structural models

Keep second- and third-ranked pockets containing at least 75% of the
residues in their corresponding original pocket. Retain all original
structures and alternative pockets that meet this minimum.


In [2]:
original_residues = originals.set_index("BaseProtein")["Number of residues"]
residue_retention = (
    models["Number of residues"]
    / models["BaseProtein"].map(original_residues)
)
retained_models = models.loc[
    models["Rank"].eq(1)
    | residue_retention.ge(MINIMUM_RESIDUE_RETENTION)
].copy()
alternatives = retained_models.loc[
    retained_models["Rank"].isin([2, 3])
].copy()
alternatives["Original pocket residues (%)"] = (
    100
    * alternatives["Number of residues"]
    / alternatives["BaseProtein"].map(original_residues)
)
eligible_proteins = set(alternatives["BaseProtein"])
complete_triplicates = set(
    alternatives.groupby("BaseProtein")["Rank"]
    .agg(lambda ranks: set(ranks) == {2, 3})
    .loc[lambda complete: complete]
    .index
)

print(f"Minimum residue retention: {100 * MINIMUM_RESIDUE_RETENTION:g}%")
print(f"Selected alternative models: {len(alternatives)}")
print(f"Proteins with alternative models: {len(eligible_proteins)}")
print(f"Complete triplicates: {len(complete_triplicates)}")
print(
    "Alternative pocket sizes: "
    f"{alternatives['Original pocket residues (%)'].min():.1f}-"
    f"{alternatives['Original pocket residues (%)'].max():.1f}% of the original"
)
print("Alternative models by rank:")
print(alternatives["Rank"].value_counts().sort_index().to_string())


Minimum residue retention: 75%
Selected alternative models: 119
Proteins with alternative models: 70
Complete triplicates: 49
Alternative pocket sizes: 75.0-216.7% of the original
Alternative models by rank:
Rank
2    58
3    61


## Compare pocket features

Feature similarity is calculated as
`100 × (1 - |original - alternative| / (|original| + |alternative|))`.
Two zero values are assigned 100% similarity. Amino-acid composition
similarity compares normalized residue frequencies.


In [3]:
FEATURE_SETS = {
    "Amino-acid features": AMINO_ACID_FEATURES,
    "All pocket features": features,
    "FS1 features": SELECTED_FEATURES,
}


def feature_similarity(original, alternative):
    original = np.asarray(original, dtype=float)
    alternative = np.asarray(alternative, dtype=float)
    denominator = np.abs(original) + np.abs(alternative)
    similarity = np.full(original.shape, 100.0)
    informative = denominator > 0
    similarity[informative] = 100 * (
        1 - np.abs(original[informative] - alternative[informative])
        / denominator[informative]
    )
    return np.clip(similarity, 0, 100)


def amino_acid_overlap(original, alternative):
    original = np.asarray(original, dtype=float)
    alternative = np.asarray(alternative, dtype=float)
    if original.sum() <= 0 or alternative.sum() <= 0:
        return np.nan
    return 100 * np.minimum(
        original / original.sum(), alternative / alternative.sum()
    ).sum()


originals_by_protein = originals.set_index("BaseProtein")
comparison_rows = []
selected_feature_rows = []

for _, alternative in alternatives.iterrows():
    protein = alternative["BaseProtein"]
    original = originals_by_protein.loc[protein]
    composition = amino_acid_overlap(
        original[AMINO_ACID_FEATURES], alternative[AMINO_ACID_FEATURES]
    )

    for name, selected_columns in FEATURE_SETS.items():
        original_values = original[selected_columns].to_numpy(dtype=float)
        alternative_values = alternative[selected_columns].to_numpy(dtype=float)
        similarity = feature_similarity(original_values, alternative_values)
        comparison_rows.append({
            "Protein": protein,
            "Rank": int(alternative["Rank"]),
            "Feature set": name,
            "Similarity (%)": float(similarity.mean()),
            "Amino-acid composition similarity (%)": float(composition),
            "Original residues": float(original["Number of residues"]),
            "Alternative residues": float(alternative["Number of residues"]),
        })

        if name == "FS1 features":
            selected_feature_rows.extend(
                {
                    "Protein": protein,
                    "Rank": int(alternative["Rank"]),
                    "Feature": feature,
                    "Original value": float(old),
                    "Alternative value": float(new),
                    "Absolute difference": float(abs(new - old)),
                    "Similarity (%)": float(score),
                }
                for feature, old, new, score in zip(
                    selected_columns, original_values, alternative_values, similarity
                )
            )

comparisons = pd.DataFrame(comparison_rows)
selected_comparisons = pd.DataFrame(selected_feature_rows)
similarity_summary = (
    comparisons.groupby("Feature set", sort=False)["Similarity (%)"]
    .agg(["count", "mean", "std"])
    .rename(columns={"count": "Comparisons", "mean": "Mean (%)", "std": "SD (%)"})
    .round(2)
)
print("Pocket-feature similarity")
print(similarity_summary.to_string())


feature_stability_rows = []
for feature in SELECTED_FEATURES:
    values = selected_comparisons.loc[
        selected_comparisons["Feature"].eq(feature)
    ]
    feature_stability_rows.append({
        "Feature": feature,
        "Original mean": values.drop_duplicates("Protein")["Original value"].mean(),
        "First alternative mean": values.loc[
            values["Rank"].eq(2), "Alternative value"
        ].mean(),
        "Second alternative mean": values.loc[
            values["Rank"].eq(3), "Alternative value"
        ].mean(),
        "Mean absolute difference": values["Absolute difference"].mean(),
        "Similarity (%)": values["Similarity (%)"].mean(),
    })
feature_stability = pd.DataFrame(feature_stability_rows).round(2)
print("\nSelected feature stability")
print(feature_stability.to_string(index=False))


Pocket-feature similarity
                     Comparisons  Mean (%)  SD (%)
Feature set                                       
Amino-acid features          119     83.84    9.95
All pocket features          119     85.65    7.50
FS1 features                 119     86.56    9.85

Selected feature stability
                      Feature  Original mean  First alternative mean  Second alternative mean  Mean absolute difference  Similarity (%)
                  Alanine - A           1.47                    1.40                     1.49                      0.76           62.04
               Methionine - M           0.51                    0.55                     0.56                      0.09           91.88
               Tryptophan - W           1.14                    1.09                     1.15                      0.24           86.55
               Isoleucine - I           2.20                    2.26                     2.21                      0.27           92.45
           

## Test classification across structural models

For each protein with an alternative model, train the final XGBoost
classifier on all other proteins in the repository's `FullDataset.csv`.
Predict the withheld original structure and its selected alternatives
using the same fitted classifier. Calculate inverse-frequency class
weights separately for each training fold.


In [4]:
def classifier():
    return XGBClassifier(
        eval_metric="logloss",
        reg_alpha=0.1,
        reg_lambda=1.0,
        verbosity=0,
        random_state=RANDOM_SEED,
        tree_method="hist",
        learning_rate=0.1,
    )


prediction_rows = []
for protein in sorted(eligible_proteins):
    training = full_dataset.loc[full_dataset["BaseProtein"].ne(protein)]
    labels = training["Cyclical"]
    if set(labels) != {0, 1}:
        raise ValueError(f"Training fold for {protein} needs both classes.")

    frequencies = labels.value_counts(normalize=True)
    sample_weights = labels.map(1 / frequencies).to_numpy()
    model = classifier().fit(
        training[SELECTED_FEATURES], labels, sample_weight=sample_weights
    )
    test_models = retained_models.loc[
        retained_models["BaseProtein"].eq(protein)
    ].sort_values("Rank")
    probabilities = model.predict_proba(test_models[SELECTED_FEATURES])[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    prediction_rows.extend(
        {
            "Protein": protein,
            "Model": model_id,
            "Rank": int(rank),
            "Observed": int(observed),
            "Predicted": int(predicted),
            "Cyclic probability": float(probability),
            "Complete triplicate": protein in complete_triplicates,
        }
        for model_id, rank, observed, predicted, probability in zip(
            test_models["Protein"],
            test_models["Rank"],
            test_models["Cyclical"],
            predictions,
            probabilities,
        )
    )

predictions = pd.DataFrame(prediction_rows)


def classification_metrics(name, group):
    observed = group["Observed"].to_numpy(dtype=int)
    predicted = group["Predicted"].to_numpy(dtype=int)
    metrics = {
        "Structural model": name,
        "Models": len(group),
        "Proteins": group["Protein"].nunique(),
        "Accuracy (%)": 100 * accuracy_score(observed, predicted),
        "Balanced accuracy (%)": 100 * balanced_accuracy_score(observed, predicted),
        "MCC": matthews_corrcoef(observed, predicted),
    }
    for label, class_name in ((0, "linear"), (1, "cyclic")):
        metrics[f"Precision ({class_name}) (%)"] = 100 * precision_score(
            observed, predicted, pos_label=label, zero_division=0
        )
        metrics[f"Recall ({class_name}) (%)"] = 100 * recall_score(
            observed, predicted, pos_label=label, zero_division=0
        )
        metrics[f"F1 ({class_name}) (%)"] = 100 * f1_score(
            observed, predicted, pos_label=label, zero_division=0
        )
    return metrics


subsets = [
    ("Original", predictions.loc[predictions["Rank"].eq(1)]),
    ("First alternative", predictions.loc[predictions["Rank"].eq(2)]),
    ("Second alternative", predictions.loc[predictions["Rank"].eq(3)]),
    ("All alternatives", predictions.loc[predictions["Rank"].isin([2, 3])]),
]
classification = pd.DataFrame(
    classification_metrics(name, group)
    for name, group in subsets
    if not group.empty
)
print("Leave-one-protein-out classification")
print(classification.round(3).to_string(index=False))

original_predictions = predictions.loc[
    predictions["Rank"].eq(1)
].set_index("Protein")
alternative_predictions = predictions.loc[
    predictions["Rank"].isin([2, 3])
].copy()
alternative_predictions["Original prediction"] = (
    alternative_predictions["Protein"].map(original_predictions["Predicted"])
)
alternative_predictions["Original probability"] = (
    alternative_predictions["Protein"].map(original_predictions["Cyclic probability"])
)
agreement = 100 * (
    alternative_predictions["Predicted"]
    .eq(alternative_predictions["Original prediction"])
    .mean()
)
probability_change = 100 * (
    alternative_predictions["Cyclic probability"]
    .sub(alternative_predictions["Original probability"])
    .abs()
    .mean()
)
print(f"\nPrediction agreement: {agreement:.1f}%")
print(f"Mean probability change: {probability_change:.2f} percentage points")


Leave-one-protein-out classification
  Structural model  Models  Proteins  Accuracy (%)  Balanced accuracy (%)   MCC  Precision (linear) (%)  Recall (linear) (%)  F1 (linear) (%)  Precision (cyclic) (%)  Recall (cyclic) (%)  F1 (cyclic) (%)
          Original      70        70        92.857                 93.571 0.847                  97.727               91.489           94.505                  84.615               95.652           89.796
 First alternative      58        58        89.655                 88.803 0.776                  91.892               91.892           91.892                  85.714               85.714           85.714
Second alternative      61        61        90.164                 88.841 0.777                  92.683               92.683           92.683                  85.000               85.000           85.000
  All alternatives     119        70        89.916                 88.837 0.777                  92.308               92.308           92.308      

## Summarize structural sensitivity


In [5]:
overview_rows = []
eligible_originals = originals.loc[originals["BaseProtein"].isin(eligible_proteins)]
model_groups = [
    ("Original", eligible_originals, None),
    ("First alternative", alternatives.loc[alternatives["Rank"].eq(2)], 2),
    ("Second alternative", alternatives.loc[alternatives["Rank"].eq(3)], 3),
    ("All alternatives", alternatives, "all"),
]

for name, group, rank in model_groups:
    if group.empty:
        continue
    subset = comparisons if rank == "all" else comparisons.loc[
        comparisons["Rank"].eq(rank)
    ] if rank is not None else comparisons.iloc[:0]
    amino_subset = subset.loc[
        subset["Feature set"].eq("Amino-acid features")
    ]
    all_subset = subset.loc[subset["Feature set"].eq("All pocket features")]
    selected_subset = subset.loc[subset["Feature set"].eq("FS1 features")]
    scores = classification.loc[
        classification["Structural model"].eq(name)
    ].iloc[0]

    overview_rows.append({
        "Structural model": name,
        "Models": len(group),
        "Proteins": group["BaseProtein"].nunique(),
        "Pocket residues mean": group["Number of residues"].mean(),
        "Pocket residues SD": group["Number of residues"].std(),
        "Amino-acid composition similarity (%)": amino_subset[
            "Amino-acid composition similarity (%)"
        ].mean(),
        "All-feature similarity (%)": all_subset["Similarity (%)"].mean(),
        "FS1-feature similarity (%)": selected_subset["Similarity (%)"].mean(),
        "Accuracy (%)": scores["Accuracy (%)"],
        "Balanced accuracy (%)": scores["Balanced accuracy (%)"],
        "MCC": scores["MCC"],
    })

overview = pd.DataFrame(overview_rows).round(3)
output_directory = DATASET_FILE.parent / "results"
output_directory.mkdir(exist_ok=True)
overview.to_csv(output_directory / "Structural_Sensitivity_Overview.csv", index=False)
feature_stability.to_csv(
    output_directory / "Selected_Feature_Stability.csv", index=False
)

print("Structural sensitivity overview")
print(overview.to_string(index=False))
print(f"\nResults saved to {output_directory}")


Structural sensitivity overview
  Structural model  Models  Proteins  Pocket residues mean  Pocket residues SD  Amino-acid composition similarity (%)  All-feature similarity (%)  FS1-feature similarity (%)  Accuracy (%)  Balanced accuracy (%)   MCC
          Original      70        70                32.229               7.039                                    NaN                         NaN                         NaN        92.857                 93.571 0.847
 First alternative      58        58                33.414               8.764                                 86.455                      85.957                      86.618        89.655                 88.803 0.776
Second alternative      61        61                33.590               7.699                                 85.067                      85.350                      86.512        90.164                 88.841 0.777
  All alternatives     119        70                33.504               8.201                      